# NYSE TAQ BBO Quote Exploration

This notebook walks through the NYSE-specific BBO (Best Bid and Offer) quote file:

- `EQY_US_TAQ_NYSE_BBO_1_20231002.gz`

This file covers **hour 1** (9:30–10:30 ET) of NYSE-listed securities on October 2, 2023.

We will:
1. Inspect the raw schema and message types.
2. Parse BBO quote updates and compute spread / mid / depth features.
3. Build symbol-minute quote bars for selected tickers.
4. Visualize intraday spread, depth, and quote-intensity profiles.
5. Compare what features we get from **trades** vs. **quotes** and what requires **both**.

**Companion notebook:** `NYSE_TAQ_Consolidated_Trades_Exploration.ipynb` covers the trade side.

**Note:** This BBO file is NYSE-only (not consolidated NBBO). The full NBBO file (`EQY_US_ALL_NBBO_20260102.gz`, ~9.5 GB) is downloading now and will give the same features across all venues.

In [1]:
from pathlib import Path
import gzip
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

DATA_DIR = Path('../data/nyse_taq_samples')
RESEARCH_DIR = Path('.')

BBO_PATH = DATA_DIR / 'EQY_US_TAQ_NYSE_BBO_1_20231002.gz'
TRADE_PATH = DATA_DIR / 'EQY_US_ALL_TRADE_20260102.gz'

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

print('BBO file exists:', BBO_PATH.exists())
if BBO_PATH.exists():
    print(f'BBO file size: {BBO_PATH.stat().st_size / 1e6:.1f} MB')
print('Consolidated trade file exists:', TRADE_PATH.exists())

BBO file exists: True
BBO file size: 1657.5 MB
Consolidated trade file exists: True


## 1. What the file is

`EQY_US_TAQ_NYSE_BBO_1_20231002.gz` is the **NYSE venue-specific BBO** (Best Bid and Offer) for hour 1 of trading on October 2, 2023.

- It contains **quote updates** (message type `140`) and some trade/correction messages.
- Each BBO update gives the current best bid price/size and best ask price/size on NYSE.
- It is **comma-delimited** (unlike the consolidated trades file which is pipe-delimited).
- There is **no header row** — the first field is the message type code.
- This is a single-exchange view; the NBBO file gives the consolidated best quotes across all venues.

In [2]:
def peek_bbo(path, n=10):
    """Read the first n raw lines from the BBO file."""
    with gzip.open(path, 'rt', encoding='latin-1', errors='replace') as f:
        return [f.readline().strip() for _ in range(n)]

raw_lines = peek_bbo(BBO_PATH, 10)
for i, line in enumerate(raw_lines):
    parts = line.split(',')
    print(f'Line {i+1} (type={parts[0]}): {line[:200]}')

Line 1 (type=3): 3,2,BKD,1,58,N,C,100,4.14,0,0,N,.0001,1
Line 2 (type=34): 34,4,00:28:41.057267806,BKD,1,P,~,,,,,,~,P
Line 3 (type=3): 3,5,AB,1,58,N,C,100,30.35,0,0,N,.0001,1
Line 4 (type=34): 34,7,00:28:41.057335483,AB,1,P,~,,,,,,~,P
Line 5 (type=3): 3,8,GJO,1,58,N,M,100,24.99,0,0,N,.0001,1
Line 6 (type=34): 34,10,00:28:41.057453967,GJO,1,P,~,,,,,,~,P
Line 7 (type=3): 3,11,GJP,1,58,N,M,100,25,0,0,N,.0001,1
Line 8 (type=34): 34,13,00:28:41.057454425,GJP,1,P,~,,,,,,~,P
Line 9 (type=3): 3,14,HOMB,1,58,N,C,100,20.94,0,0,N,.0001,1
Line 10 (type=34): 34,16,00:28:41.057510550,HOMB,1,P,~,,,,,,~,P


## 2. Message types and column layout

The BBO file is a mixed-message feed. The first comma-separated field is the message type:

| Code | Meaning | Key fields |
|---|---|---|
| `140` | BBO quote update | seq, time, symbol, quote_cond, bid_px, bid_sz, ask_px, ask_sz, flag |
| `3` | Trade | seq, symbol, ..., sale_cond, size, price |
| `34` | Trade correction | seq, timestamp, symbol, ... |

For feature engineering, the `140` messages are the main payload. Each one is a snapshot of the NYSE best bid/ask at that moment.

In [3]:
# Count message types in a sample
msg_types = Counter()
max_scan = 500_000
with gzip.open(BBO_PATH, 'rt', encoding='latin-1', errors='replace') as f:
    for i, line in enumerate(f):
        parts = line.strip().split(',')
        if parts:
            msg_types[parts[0]] += 1
        if i >= max_scan:
            break

print(f'Messages scanned: {sum(msg_types.values()):,}')
print('Message type distribution:')
for mt, count in msg_types.most_common():
    print(f'  Type {mt:>3s}: {count:>8,}')

Messages scanned: 500,001
Message type distribution:
  Type 140:  470,081
  Type  34:   24,127
  Type   3:    5,793


## 3. Parse BBO quotes and compute spread features

We stream the file, extract all `140` (BBO quote) messages, and compute:
- **Bid price / size** and **ask price / size**
- **Mid price** = (bid + ask) / 2
- **Spread** = ask - bid (in price and basis points)
- **Depth imbalance** = (bid_sz - ask_sz) / (bid_sz + ask_sz)
- **Quote time** converted to minute-of-day

In [ ]:
def parse_time_ns(time_str):
    """Convert HHMMSSNNNNNNNNN string to minute-of-day integer."""
    try:
        hours = int(time_str[:2])
        minutes = int(time_str[2:4])
        return hours * 60 + minutes
    except Exception:
        return np.nan

def parse_bbo_quotes(path, max_rows=None):
    """Stream the BBO file and extract all type-140 quote messages."""
    rows = []
    with gzip.open(path, 'rt', encoding='latin-1', errors='replace') as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) < 9 or parts[0] != '140':
                continue
            try:
                minute = parse_time_ns(parts[2])
                bid_px = float(parts[5]) if parts[5] else np.nan
                bid_sz = int(parts[6]) if parts[6] else 0
                ask_px = float(parts[7]) if parts[7] else np.nan
                ask_sz = int(parts[8]) if parts[8] else 0
            except (ValueError, IndexError):
                continue
            if np.isnan(minute):
                continue
            rows.append({
                'symbol': parts[3],
                'time_bin': int(minute),
                'quote_cond': parts[4].strip(),
                'bid_px': bid_px,
                'bid_sz': bid_sz,
                'ask_px': ask_px,
                'ask_sz': ask_sz,
            })
            if max_rows and len(rows) >= max_rows:
                break
    df = pd.DataFrame(rows)
    # Filter to valid two-sided quotes
    valid = (df['bid_px'] > 0) & (df['ask_px'] > 0) & (df['ask_px'] >= df['bid_px'])
    df = df.loc[valid].copy()
    df['mid'] = (df['bid_px'] + df['ask_px']) / 2
    df['spread'] = df['ask_px'] - df['bid_px']
    df['spread_bps'] = df['spread'] / df['mid'] * 10000
    df['depth_imbalance'] = (df['bid_sz'] - df['ask_sz']) / (df['bid_sz'] + df['ask_sz'])
    df['total_depth'] = df['bid_sz'] + df['ask_sz']
    return df

bbo_df = parse_bbo_quotes(BBO_PATH)
print(f'Total BBO quotes parsed: {len(bbo_df):,}')
print(f'Unique symbols: {bbo_df["symbol"].nunique()}')
bbo_df.head()

In [7]:
bbo_df = parse_bbo_quotes(BBO_PATH)
print(f'Total BBO quotes parsed: {len(bbo_df):,}')
print(f'Unique symbols: {bbo_df["symbol"].nunique()}')
bbo_df.head()

KeyError: 'bid_px'

## 4. Spread and liquidity statistics

A first look at the distribution of spreads and depth across all quotes in the sample.

In [5]:
print('Spread (bps) summary:')
print(bbo_df['spread_bps'].describe())
print()
print('Depth (total shares at best bid+ask) summary:')
print(bbo_df['total_depth'].describe())
print()
print('Depth imbalance summary (-1 = all ask, +1 = all bid):')
print(bbo_df['depth_imbalance'].describe())

Spread (bps) summary:


NameError: name 'bbo_df' is not defined

In [ ]:
# Top symbols by quote frequency
top_quote_symbols = bbo_df['symbol'].value_counts().head(20)
print('Top 20 symbols by quote update count:')
print(top_quote_symbols)

## 5. Build symbol-minute quote bars

Aggregate quote updates into one-minute bars per symbol. This is the quote-side analogue of the trade-minute bars in the companion notebook.

In [ ]:
def aggregate_quote_bars(df):
    """Aggregate BBO quotes into symbol-minute bars."""
    bars = df.groupby(['symbol', 'time_bin']).agg(
        quote_count=('mid', 'count'),
        avg_spread_bps=('spread_bps', 'mean'),
        med_spread_bps=('spread_bps', 'median'),
        max_spread_bps=('spread_bps', 'max'),
        avg_spread=('spread', 'mean'),
        avg_mid=('mid', 'mean'),
        open_mid=('mid', 'first'),
        close_mid=('mid', 'last'),
        high_mid=('mid', 'max'),
        low_mid=('mid', 'min'),
        avg_bid_sz=('bid_sz', 'mean'),
        avg_ask_sz=('ask_sz', 'mean'),
        avg_total_depth=('total_depth', 'mean'),
        avg_depth_imbalance=('depth_imbalance', 'mean'),
    ).reset_index()
    
    # Mid-price return per minute
    bars['mid_return'] = bars.groupby('symbol').apply(
        lambda g: np.log(g['close_mid']) - np.log(g['close_mid'].shift(1))
    ).reset_index(level=0, drop=True)
    
    bars['mid_range'] = np.log(bars['high_mid']) - np.log(bars['low_mid'])
    bars['hour'] = bars['time_bin'] // 60
    bars['minute_of_hour'] = bars['time_bin'] % 60
    
    return bars

quote_bars = aggregate_quote_bars(bbo_df)
quote_bars.head()

## 6. Visualize intraday quote profiles

These are the quote-side analogues of the trade-side plots in the companion notebook.

In [ ]:
# Pick top symbols by quote count for visualization
viz_symbols = top_quote_symbols.head(8).index.tolist()
viz_bars = quote_bars[quote_bars['symbol'].isin(viz_symbols)].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Spread profile
ax = axes[0, 0]
for symbol, group in viz_bars.groupby('symbol'):
    group = group.sort_values('time_bin')
    ax.plot(group['time_bin'], group['avg_spread_bps'].rolling(5, min_periods=1).mean(), label=symbol, alpha=0.8)
ax.set_title('Intraday spread profile (5-min MA, bps)')
ax.set_xlabel('Minute of day')
ax.set_ylabel('Spread (bps)')
ax.legend(fontsize=7)

# Quote intensity profile
ax = axes[0, 1]
for symbol, group in viz_bars.groupby('symbol'):
    group = group.sort_values('time_bin')
    ax.plot(group['time_bin'], group['quote_count'].rolling(5, min_periods=1).mean(), label=symbol, alpha=0.8)
ax.set_title('Intraday quote intensity (5-min MA)')
ax.set_xlabel('Minute of day')
ax.set_ylabel('Quote updates per minute')
ax.legend(fontsize=7)

# Depth profile
ax = axes[1, 0]
for symbol, group in viz_bars.groupby('symbol'):
    group = group.sort_values('time_bin')
    ax.plot(group['time_bin'], group['avg_total_depth'].rolling(5, min_periods=1).mean(), label=symbol, alpha=0.8)
ax.set_title('Intraday depth profile (5-min MA, shares)')
ax.set_xlabel('Minute of day')
ax.set_ylabel('Total depth (bid+ask shares)')
ax.legend(fontsize=7)

# Depth imbalance profile
ax = axes[1, 1]
for symbol, group in viz_bars.groupby('symbol'):
    group = group.sort_values('time_bin')
    ax.plot(group['time_bin'], group['avg_depth_imbalance'].rolling(5, min_periods=1).mean(), label=symbol, alpha=0.8)
ax.set_title('Intraday depth imbalance (5-min MA)')
ax.set_xlabel('Minute of day')
ax.set_ylabel('Depth imbalance (-1=ask heavy, +1=bid heavy)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 7. Daily summary per symbol (quote side)

In [ ]:
quote_summary = quote_bars.groupby('symbol').agg(
    total_quotes=('quote_count', 'sum'),
    avg_spread_bps=('avg_spread_bps', 'mean'),
    med_spread_bps=('med_spread_bps', 'median'),
    avg_depth=('avg_total_depth', 'mean'),
    avg_depth_imbalance=('avg_depth_imbalance', 'mean'),
    mid_open=('open_mid', 'first'),
    mid_close=('close_mid', 'last'),
).reset_index()

quote_summary['mid_return_pct'] = 100 * np.log(quote_summary['mid_close'] / quote_summary['mid_open'])
quote_summary = quote_summary.sort_values('total_quotes', ascending=False)
quote_summary.head(20)

## 8. Feature comparison: trades vs. quotes vs. both

This is the key table. It shows what features each data source supports.

### Features from trades alone (`EQY_US_ALL_TRADE_20260102.gz`)

| Feature | Description |
|---|---|
| Volume | Total shares traded per minute/hour/day |
| Dollar volume | Volume × price |
| Trade count | Number of prints per minute |
| Average trade size | Volume / trade count |
| VWAP | Volume-weighted average price |
| OHLC prices | Open/high/low/close per bar |
| Returns | Log returns from close-to-close |
| Realized variance | Sum of squared returns |
| High-low range | Log(high) - log(low) |
| Intraday volume profile | 390-dim vector of minute volume |
| Intraday volatility profile | 390-dim vector of realized variance |
| Trade-size distribution | Histogram of trade sizes |
| Venue market share | Fraction of volume by exchange code |
| Tick-test trade signing | Buy/sell initiation via tick rule |

### Features from BBO/NBBO quotes alone

| Feature | Description |
|---|---|
| Bid/ask spread | ask - bid (price and bps) |
| Mid price | (bid + ask) / 2 |
| Mid-price returns | Log returns of mid price |
| Quote intensity | Number of quote updates per minute |
| Depth (total) | bid_sz + ask_sz |
| Depth imbalance | (bid_sz - ask_sz) / (bid_sz + ask_sz) |
| Spread volatility | Std of spread within each minute |
| Quote-to-trade ratio | Quote updates per trade (needs trade count) |
| Intraday spread profile | 390-dim vector of minute spread |
| Intraday depth profile | 390-dim vector of minute depth |
| Liquidity regime | Clustering on spread + depth + intensity |
| Quote-based volatility | Mid-price realized variance |

### Features requiring BOTH trades and quotes

| Feature | Description |
|---|---|
| Effective spread | 2 × |trade_price - mid| / mid |
| Realized spread | Effective spread measured after a short delay |
| Price impact | Signed volume → future mid return |
| Order flow imbalance (OFI) | Signed volume using quote-side classification |
| Lee-Ready trade signing | Classify trades as buy/sell using quote comparison |
| Quote-to-trade ratio | Quote updates per trade |
| Trade arrival vs. quote changes | How fast quotes react to trades |
| Adverse selection | Component of spread related to informed flow |
| Kyle's lambda | Price impact per unit of signed volume |
| Amihud illiquidity | |return| / dollar volume |
| VPIN | Volume-synchronized probability of informed trading |

### Dimension-reduction candidates

Each of these intraday curves can be a high-dimensional vector for PCA / Isomap / UMAP:

| Curve | Source | Dimensions |
|---|---|---|
| Volume profile | Trades | 390 (minutes) |
| Volatility profile | Trades | 390 |
| Spread profile | Quotes | 390 |
| Depth profile | Quotes | 390 |
| Depth imbalance profile | Quotes | 390 |
| Quote intensity profile | Quotes | 390 |
| Effective spread profile | Both | 390 |
| OFI profile | Both | 390 |

## 9. Demo: PCA vs. Isomap on intraday spread curves

Same idea as the trade-side notebook, but now using the **intraday spread profile** as the high-dimensional vector.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap

# Use symbols with enough quote activity
active_symbols = top_quote_symbols.head(15).index.tolist()
active_bars = quote_bars[quote_bars['symbol'].isin(active_symbols)]

spread_curve = active_bars.pivot_table(
    index='symbol', columns='time_bin', values='avg_spread_bps', fill_value=0
)

if len(spread_curve) >= 3:
    X = spread_curve.values
    X_scaled = StandardScaler().fit_transform(X)

    pca_emb = PCA(n_components=2, svd_solver='full').fit_transform(X_scaled)
    isomap_emb = Isomap(
        n_neighbors=min(3, len(spread_curve) - 1), n_components=2
    ).fit_transform(X_scaled)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for i, symbol in enumerate(spread_curve.index):
        axes[0].scatter(pca_emb[i, 0], pca_emb[i, 1], s=80)
        axes[0].text(pca_emb[i, 0], pca_emb[i, 1], symbol, fontsize=8)
        axes[1].scatter(isomap_emb[i, 0], isomap_emb[i, 1], s=80)
        axes[1].text(isomap_emb[i, 0], isomap_emb[i, 1], symbol, fontsize=8)
    axes[0].set_title('PCA on intraday spread curves')
    axes[1].set_title('Isomap on intraday spread curves')
    axes[0].set_xlabel('PC 1')
    axes[0].set_ylabel('PC 2')
    axes[1].set_xlabel('Isomap 1')
    axes[1].set_ylabel('Isomap 2')
    plt.tight_layout()
    plt.show()
else:
    print(f'Not enough symbols with spread data for DR demo (found {len(spread_curve)})')

## 10. Main takeaways

1. The BBO file gives us the **quote side** of microstructure: spreads, depth, mid-price dynamics, and quote intensity.
2. From quotes alone we can build intraday profiles for spread, depth, and depth imbalance — all candidates for nonlinear dimension reduction.
3. The richest features come from **combining trades and quotes**: effective spread, price impact, order flow imbalance, and trade signing.
4. This NYSE BBO file is limited to one exchange and one hour. The full NBBO file (`EQY_US_ALL_NBBO_20260102.gz`, downloading now) will give the same features across all venues for a full day.
5. Once both files are available for the same date, we can build a unified feature set and run dimension-reduction experiments on combined trade + quote curves.